# NS08 Tutorial A - Configuration Model and Degree-Preserving Null Models

**Lecture:** NS08 - Network Models

**Reading:** Newman, *Networks*, Section on the configuration model

## Learning goals
- Understand the configuration model as a **degree-based null model**.
- Translate the lecture idea of **stubs** into explicit computational steps.
- Compare an observed network to randomized baselines that preserve degree information.
- Test which properties are explained by the degree sequence alone, and which are not.


In [ ]:
from netsci_utils import *
import pandas as pd

set_seeds()
%matplotlib inline


def graph_summary(G, name):
    degrees = np.array([degree for _, degree in G.degree()], dtype=float)
    largest_nodes = max(nx.connected_components(G), key=len)
    largest = G.subgraph(largest_nodes).copy()
    return {
        'network': name,
        'n': G.number_of_nodes(),
        'm': G.number_of_edges(),
        '<k>': degrees.mean(),
        'max k': degrees.max(),
        'kappa': heterogeneity_kappa(G),
        'avg clustering': nx.average_clustering(G),
        'largest component fraction': largest_component_fraction(G),
        'avg path length in LCC': nx.average_shortest_path_length(largest),
    }


def stub_pairing_from_degree_sequence(degree_sequence, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    stubs = []
    for node, degree in enumerate(degree_sequence):
        stubs.extend([node] * degree)
    stubs = np.array(stubs)
    rng.shuffle(stubs)
    pairs = list(zip(stubs[::2], stubs[1::2]))
    return stubs, pairs


def simple_configuration_sample(G, seed=RANDOM_SEED):
    degree_sequence = [degree for _, degree in G.degree()]
    H = nx.configuration_model(degree_sequence, seed=seed)
    H = nx.Graph(H)
    H.remove_edges_from(nx.selfloop_edges(H))
    return H


def degree_preserving_rewire(G, nswap_factor=5, seed=RANDOM_SEED):
    H = G.copy()
    nswap = nswap_factor * H.number_of_edges()
    nx.double_edge_swap(H, nswap=nswap, max_tries=20 * nswap, seed=seed)
    return H

---
## 1. Observed structure first: the US airport network

We return to **OpenFlights USA** because it is a good NS08 test case.
The degree sequence is highly heterogeneous, but the lecture question is more specific:

> If we preserve only the degree sequence, how much of the observed structure remains?


In [ ]:
openflights = load_openflights_usa()
geo_pos = positions_from_node_attributes(openflights)
degree_centrality = dict(openflights.degree())

plot_metric(
    openflights,
    degree_centrality,
    pos=geo_pos,
    with_labels=False,
    colorbar=False,
    min_node_size_px=6,
    max_node_size_px=18,
    width=0.5,
    edge_color='#BBBBBB',
    title='OpenFlights USA: degree heterogeneity in the observed network',
)

observed_summary = pd.DataFrame([graph_summary(openflights, 'Observed OpenFlights USA')])
print(observed_summary.round(3).to_string(index=False))


---
## 2. The stub idea on a toy degree sequence

The configuration model starts from a degree sequence and assigns each node that many **stubs**. Then it pairs stubs uniformly at random.

This is the computational meaning of the lecture algorithm.


In [ ]:
toy_degree_sequence = [3, 3, 2, 2, 1, 1]
stubs, pairs = stub_pairing_from_degree_sequence(toy_degree_sequence)

print('Toy degree sequence:')
for node, degree in enumerate(toy_degree_sequence):
    print(f'  node {node}: degree {degree}')

print('\nStub list after shuffling:')
print(stubs.tolist())

print('\nStub pairings:')
for edge in pairs:
    print(edge)

toy_multigraph = nx.MultiGraph()
toy_multigraph.add_nodes_from(range(len(toy_degree_sequence)))
toy_multigraph.add_edges_from(pairs)
toy_simple = nx.Graph(toy_multigraph)
toy_simple.remove_edges_from(nx.selfloop_edges(toy_simple))

toy_pos = nx.spring_layout(toy_simple, seed=RANDOM_SEED)
plot_graph(
    toy_simple,
    pos=toy_pos,
    with_labels=True,
    node_size=700,
    title='One simple graph obtained from a random stub pairing',
)


**Interpretation.**
- The degree sequence constrains how many stubs each node contributes.
- The actual pairing is random, so the model destroys most higher-order structure.
- That is exactly why it is a useful null model.


---
## 3. Two degree-based nulls for the observed network

We build two null baselines:
- a **configuration-model sample** from the observed degree sequence,
- a **degree-preserving rewired graph** obtained by repeated double-edge swaps.

The first is the classic stub model. The second keeps the graph simple and preserves each node degree exactly.


In [ ]:
config_sample = simple_configuration_sample(openflights, seed=RANDOM_SEED)
rewired_sample = degree_preserving_rewire(openflights, nswap_factor=5, seed=RANDOM_SEED)

summary_df = pd.DataFrame([
    graph_summary(openflights, 'Observed'),
    graph_summary(config_sample, 'Configuration sample'),
    graph_summary(rewired_sample, 'Degree-preserving rewire'),
])
print(summary_df.round(3).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=FIGURE_SIZE)
for name, graph, color in [
    ('Observed', openflights, CATEGORY_PALETTE['blue']),
    ('Configuration sample', config_sample, CATEGORY_PALETTE['orange']),
    ('Degree-preserving rewire', rewired_sample, CATEGORY_PALETTE['green']),
]:
    plot_ccdf(
        [degree for _, degree in graph.degree()],
        ax=ax,
        label=name,
        color=color,
        markersize=3,
    )
ax.set_title('Degree CCDF under observed and degree-based null graphs')
ax.legend(frameon=False)
plt.show()

original_degrees = sorted(degree for _, degree in openflights.degree())
rewired_degrees = sorted(degree for _, degree in rewired_sample.degree())
config_degrees = sorted(degree for _, degree in config_sample.degree())

print(f"Degree sequence preserved exactly by rewiring: {original_degrees == rewired_degrees}")
print(f"Degree sequence preserved exactly by simplified configuration sample: {original_degrees == config_degrees}")


**Interpretation.**
- The edge-swap randomization keeps the degree sequence **exactly**.
- The simplified configuration sample is more radical: after removing self-loops and parallel edges, some degrees can change.
- In both cases, clustering and assortative structure are treated as hypotheses to test, not as assumptions.


---
## 4. Null distribution of clustering under degree-preserving rewiring

If degree alone explained the observed clustering, then the observed value should look typical under repeated degree-preserving randomizations.


In [ ]:
clustering_null = []
assortativity_null = []
for seed in range(15):
    sample = degree_preserving_rewire(openflights, nswap_factor=5, seed=RANDOM_SEED + seed)
    clustering_null.append(nx.average_clustering(sample))
    assortativity_null.append(nx.degree_assortativity_coefficient(sample))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(clustering_null, bins=8, color=CATEGORY_PALETTE['blue'], edgecolor='white')
axes[0].axvline(nx.average_clustering(openflights), color=CATEGORY_PALETTE['red'], linewidth=2, linestyle='--')
style_axis(
    axes[0],
    title='Clustering under degree-preserving rewiring',
    xlabel='Average clustering',
    ylabel='Count',
)

axes[1].hist(assortativity_null, bins=8, color=CATEGORY_PALETTE['green'], edgecolor='white')
axes[1].axvline(nx.degree_assortativity_coefficient(openflights), color=CATEGORY_PALETTE['red'], linewidth=2, linestyle='--')
style_axis(
    axes[1],
    title='Assortativity under degree-preserving rewiring',
    xlabel='Degree assortativity',
    ylabel='Count',
)

plt.show()

print(f"Observed clustering     : {nx.average_clustering(openflights):.4f}")
print(f"Null mean clustering    : {np.mean(clustering_null):.4f}")
print(f"Observed assortativity  : {nx.degree_assortativity_coefficient(openflights):.4f}")
print(f"Null mean assortativity : {np.mean(assortativity_null):.4f}")


## Takeaway

- The configuration model is a **degree-based null**, not a growth mechanism.
- Degree-preserving rewiring is the simple-graph version of the same idea.
- If a property remains unusual after degree-preserving randomization, then the degree sequence alone does **not** explain it.
- In a transportation network, this is how we ask whether route clustering reflects more than just the presence of high-degree hubs.
